In [ ]:
get_ipython().system('hostname')

In [ ]:
from photometry.models.baselines import LambertianModel
from photometry.fitting.least_sq import LeastSquaresFitter
from photometry.core.types import GeometryBatch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np 
import pandas as pd
from pathlib import Path
import rasterio
import spiceypy as spice
import plotly.graph_objects as go
from scipy.optimize import curve_fit
import duckdb
import os
print(os.getcwd())

In [ ]:
# Set up paths and load phase-curve data for all phases

project_root = Path.cwd().resolve()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent



parquet_path_survey_gaskell_dsk256_110825 = project_root / "data" / "geometry" / "gaskell_dsk256_110825" / "survey"/"*.parquet"

parquet_path_hamo_gaskell_dsk256_110825 = project_root / "data" / "geometry" / "gaskell_dsk256_110825" / "hamo"/"*.parquet"

parquet_path_lamo_gaskell_dsk256_110825 = project_root / "data" / "geometry" / "gaskell_dsk256_110825" / "lamo"/"*.parquet"

binned_parquet_path_survey_gaskell_dsk256_110825 = project_root / "data" / "golden" / "survey_binned_dsk256_110825_range80.parquet"



dtm_path = project_root / "data" / "dtm" / "DTM_VESTA_93M.TIF"

dsk_path = project_root / "data" / "spice_kernels" / "vesta_gaskell_256_110825.bds"

In [ ]:
survey_df = pd.read_parquet(binned_parquet_path_survey_gaskell_dsk256_110825)
display(survey_df)

print(survey_df.columns.tolist())

In [ ]:
df = survey_df.copy()


df["mu0_true"] = np.cos(np.deg2rad(df["mean_incidence"]))
df["mu_true"] = np.cos(np.deg2rad(df["mean_emission"]))

# Filter out high phases if needed
df = df[df["alpha_grid"] < 80]

In [ ]:
display(df)

In [ ]:
# --- Minnaert model: r = A * mu0^k * mu^(k-1) ---
def minnaert(X, A, k):
    mu0, mu = X
    return A * np.power(mu0, k) * np.power(mu, k - 1)

In [ ]:
# --- Fit A(alpha), k(alpha) per phase bin ---
results = []

for alpha_bin, g in df.groupby("alpha_grid"):

    if len(g) < 5:  # skip sparse bins, need enough i/e coverage to constrain both params
        continue

    X = (g["mu0_true"].to_numpy(), g["mu_true"].to_numpy())
    y = g["mean_iof"].to_numpy()

    sigma = 1 / np.sqrt(g["n_pixels"].to_numpy())

    try:
        popt, pcov = curve_fit(
            minnaert, X, y, 
            p0=[0.2, 0.5], 
            sigma=sigma,
            absolute_sigma=False,
            method="trf", 
            bounds=([0, 0], [2, 2])
        )
        perr = np.sqrt(np.diag(pcov))
        resid = y - minnaert(X, *popt)

        results.append({
            "alpha_bin": alpha_bin, 
            "A": popt[0], "A_err": perr[0],
            "k": popt[1], "k_err": perr[1],
            "n_bins": len(g), 
            "total_pixels": g["n_pixels"].sum(),
            "weighted_rms": np.sqrt(np.average(resid**2, weights=g["n_pixels"].to_numpy())),
        })
    except RuntimeError:
        print(f"Fit failed for alpha_bin={alpha_bin}")

minnaert_results = pd.DataFrame(results).sort_values("alpha_bin")
display(minnaert_results)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].errorbar(minnaert_results["alpha_bin"], minnaert_results["A"],
                 yerr=minnaert_results["A_err"], fmt="o", ms=4, color="blue")
axes[0].set_xlabel("Phase Angle Grid (deg)")
axes[0].set_ylabel("Minnaert Albedo $A(\\alpha)$")
axes[0].set_title("Empirical Phase Curve")

axes[1].errorbar(minnaert_results["alpha_bin"], minnaert_results["k"],
                 yerr=minnaert_results["k_err"], fmt="o", ms=4, color="darkorange")
axes[1].axhline(1.0, color="gray", ls="--", lw=1, label="Lambertian (k=1)")
axes[1].axhline(0.5, color="gray", ls=":", lw=1, label="Lommel-Seeliger (k=0.5)")
axes[1].set_xlabel("Phase Angle Grid (deg)")
axes[1].set_ylabel("Minnaert Exponent $k(\\alpha)$")
axes[1].set_title("Limb Darkening Behavior")
axes[1].legend()

plt.tight_layout()
plt.show()